# Phase 21 — do the Coverage rules fix the omissions?

Phase 20 measured the system's dominant failure as **omission, not hallucination**: 18 of 19 relevancy penalties were "omits X". It was **not** a retrieval failure — 9 of the 10 articles the judge called missing were already in the context handed to the model, and the gold article sat at **rank 1 in 16 of the 19** penalised questions. Measured directly: answers named a mean of **1.22 of the 3.87** distinct articles offered (32%), and **48 of 60 cited exactly one**.

That behaviour was *instructed*. The Phase 20 prompt said "Keep it brief", "answer in a sentence or two", and "do not reason about how two provisions relate" — three pushes toward short single-source answers, with no instruction anywhere to check every document.

## What changed (two things, measured by different instruments)

**1. Prompt: 8 rules → 12**, adding a Coverage section (rules 4–6). The guardrail — *state each provision on its own terms, do not explain how they combine* — sits immediately after the coverage rules, because Phase 12b's regression happened at exactly that spot: a "cover every consequence" rule made the model invent a connective claim between two articles (q29, faithfulness 5 → 3). **Listing provisions separately is not inference across provisions.** Rules 8, 9 and the new clause in rule 3 target the other counted defects: 6 of 14 faithfulness penalties were qualified rules stated flat, and 3 answers named the wrong law.

**2. Code: `_format_context` is now bilingual.** It labelled every document with English-only law names, so an Arabic question got Arabic article text under an English heading and the model cited what it was shown — 8 of 30 Arabic answers wrote "Article 65" instead of "المادة (65)". This is a **bug fix, not a variable under test**, and it is scored by a deterministic regex rather than by the judge, so it cannot contaminate the coverage result.

## Design: paired, on identical context

Retrieval did not change between the two arms — same index, same fusion, same `top_k`. So this notebook **loads Phase 20's saved retrieval verbatim** rather than re-running it. Two consequences, both good: the arms see byte-identical documents (so any difference is attributable to the prompt, not to Pinecone's approximate-NN drifting between calls), and retrieval costs nothing.

## Pre-committed reading of the result

Written before the run, so the conclusion can't be shaped afterwards. The Phase 20 baseline is: articles-cited **1.22**, faithfulness **4.70**, relevancy **4.55**, English-cited Arabic answers **8/30**, wrong-law **3/60**.

- **The coverage rules worked** if articles-cited rises *and* relevancy rises, with faithfulness not falling by more than the Phase 12b noise floor (~0.1).
- **They backfired** if faithfulness drops materially — the Phase 12b failure mode returning, i.e. the model padding with claims the documents don't support.
- **They did nothing** if articles-cited is flat. In that case the instruction did not take, and the next move is structured output (one line per document) rather than a stronger instruction or a bigger model.

**Known limit:** one run per arm, so the judge's own noise floor is not re-measured here. Phase 12b put it near 0.1 on these scales. Movements smaller than that are not readable from this notebook alone, and the pre-committed thresholds above are set accordingly.

In [1]:
import json
import os
import re as _re
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from src.generation import (get_production_generator, SYSTEM_PROMPT, ARTICLE_NUM_RE,
                            detect_language, _format_context)

EVAL_PATH  = PROJECT / "eval" / "eval_v3.jsonl"
# Phase 20 artefacts -- the baseline arm. Read only; never written by this notebook.
BASE_RETR  = PROJECT / "eval" / "final_eval_v3_retrieval.csv"
BASE_GEN   = PROJECT / "eval" / "final_eval_v3_generations.csv"
BASE_JUDGE = PROJECT / "eval" / "final_eval_v3_judge.csv"
# Phase 21 artefacts.
GEN_PATH   = PROJECT / "eval" / "phase21_generations.csv"
JUDGE_PATH = PROJECT / "eval" / "phase21_judge.csv"

ev = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

retr = pd.read_csv(BASE_RETR, encoding="utf-8-sig")
retr["chunk_ids"] = retr["chunk_ids"].apply(json.loads)

chunks_tbl = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
chunks_tbl["article_number"] = chunks_tbl["article_number"].astype(str)
BY_ID = {c.chunk_id: c for c in chunks_tbl.itertuples()}


def chunks_for(qid):
    ids = retr.loc[retr.question_id == qid, "chunk_ids"].iloc[0]
    return [{"chunk_id": i, "law": BY_ID[i].law, "language": BY_ID[i].language,
             "article_number": BY_ID[i].article_number, "text": BY_ID[i].text} for i in ids]


n_rules = len(_re.findall(r"^\d+\.", SYSTEM_PROMPT, _re.MULTILINE))
assert n_rules == 12, f"expected the 12-rule Phase 21 prompt, found {n_rules}"
assert "Keep it brief" not in SYSTEM_PROMPT, "still on the Phase 20 prompt"
assert "Coverage" in SYSTEM_PROMPT

print(f"eval: {len(ev)} questions | retrieval reused from {BASE_RETR.name} ({len(retr)} rows)")
print(f"prompt under test: {n_rules} rules, {len(SYSTEM_PROMPT)} chars")
print("\nArabic context now renders as:")
print("\n".join(_format_context(chunks_for("q16")[:1], "ar").splitlines()[:4]))

eval: 60 questions | retrieval reused from final_eval_v3_retrieval.csv (60 rows)
prompt under test: 12 rules, 2640 chars

Arabic context now renders as:
[وثيقة 1]
القانون: قانون الضمان الاجتماعي
المادة: 24
اللغة: ar


## 1. Generate with the Phase 21 prompt

Same retrieved documents as Phase 20, new prompt, new bilingual context labels. Answers flush to disk per question and the loop resumes, so a crash never costs a second pass.

In [2]:
gen_rows = []
if GEN_PATH.exists():
    gen_rows = pd.read_csv(GEN_PATH, encoding="utf-8-sig").to_dict("records")
done = {r["question_id"] for r in gen_rows}
print(f"{len(done)} already generated, {len(ev) - len(done)} to go")

if len(done) < len(ev):
    generator = get_production_generator()
    for q in ev.itertuples(index=False):
        if q.question_id in done:
            continue
        res = generator.generate(q.question_text, chunks_for(q.question_id))
        gen_rows.append({
            "question_id": q.question_id,
            "question_text": q.question_text,
            "question_language": q.question_language,
            "answer": res["answer"],
            "citations": json.dumps(res["citations"], ensure_ascii=False),
        })
        pd.DataFrame(gen_rows).to_csv(GEN_PATH, index=False, encoding="utf-8-sig")
        print(f"  generated {len(gen_rows)}/{len(ev)}", end="\r")

gen = pd.DataFrame(gen_rows)
print(f"\n{len(gen)} answers ready")

60 already generated, 0 to go

60 answers ready


## 2. Free checks — no judge needed

Everything here is computed from the saved answer text. These decide whether the judge pass is even worth paying for: if **articles cited** has not moved, the coverage instruction did not take, and no amount of judging will change that conclusion.

In [3]:
base_gen = pd.read_csv(BASE_GEN, encoding="utf-8-sig")


def cited_of(answer):
    return {a or b for a, b in ARTICLE_NUM_RE.findall(str(answer))}


def profile(g, label):
    rows = []
    for q in ev.itertuples(index=False):
        ans = str(g.loc[g.question_id == q.question_id, "answer"].iloc[0])
        ch = chunks_for(q.question_id)
        offered = {(c["law"], c["article_number"]) for c in ch}
        offered_nums = {a for _, a in offered}
        in_ctx = {c["article_number"] for c in ch}
        want = {str(a) for a in q.answer_article_numbers}
        cited = cited_of(ans)
        rows.append({
            "question_id": q.question_id,
            "question_language": q.question_language,
            "arm": label,
            "offered": len(offered),
            "cited": len(cited & offered_nums),
            "correct_article_cited": bool(cited & want),
            "all_correct_cited": want <= cited,
            "fabricated": sorted(cited - in_ctx),
            "chars": len(ans),
            "eng_cite_in_ar": q.question_language == "ar" and bool(_re.search(r"\bArticle\b", ans)),
        })
    return pd.DataFrame(rows)


LAW_MARKERS = {"ssl": ["Social Security", "الضمان الاجتماعي"],
               "labor96": ["Labor Law", "Labour Law", "قانون العمل"],
               "prof19": ["Professional Work", "العمل المهني"]}


def wrong_law(g):
    bad = []
    for q in ev.itertuples(index=False):
        a = str(g.loc[g.question_id == q.question_id, "answer"].iloc[0])
        named = {law for law, ms in LAW_MARKERS.items() if any(m in a for m in ms)}
        if named and q.answer_law not in named:
            bad.append(q.question_id)
    return bad


p20, p21 = profile(base_gen, "phase20"), profile(gen, "phase21")

print("=" * 74)
print("COVERAGE -- did the model start using more of its context?")
print("=" * 74)
print(f"{'':28s} {'Phase 20':>10s} {'Phase 21':>10s} {'delta':>9s}")
for lbl, col, fmt in [("mean distinct articles cited", "cited", "{:.2f}"),
                      ("mean answer length (chars)", "chars", "{:.0f}")]:
    a, b = p20[col].mean(), p21[col].mean()
    print(f"{lbl:28s} {fmt.format(a):>10s} {fmt.format(b):>10s} {b - a:>+9.2f}")
one20, one21 = (p20.cited <= 1).sum(), (p21.cited <= 1).sum()
print(f"{'answers citing <=1 article':28s} {one20:>10d} {one21:>10d} {one21 - one20:>+9d}")

# Per-question paired movement -- the deterministic part of the comparison.
m = p20[["question_id", "cited"]].merge(p21[["question_id", "cited"]],
                                        on="question_id", suffixes=("_20", "_21"))
up, down = (m.cited_21 > m.cited_20).sum(), (m.cited_21 < m.cited_20).sum()
print(f"\nper question: cites more {up}, cites fewer {down}, unchanged {len(m) - up - down}")

print()
print("=" * 74)
print("CITATION DEFECTS")
print("=" * 74)
n_ar = (ev.question_language == "ar").sum()
print(f"{'Arabic answers citing in English':38s} "
      f"{p20.eng_cite_in_ar.sum()}/{n_ar}  ->  {p21.eng_cite_in_ar.sum()}/{n_ar}")
w20, w21 = wrong_law(base_gen), wrong_law(gen)
print(f"{'answers naming only the wrong law':38s} {len(w20)}/60  ->  {len(w21)}/60   "
      f"{w20} -> {w21}")

print()
print("=" * 74)
print("REGEX PROXY")
print("=" * 74)
for lbl, col in [("correct article cited", "correct_article_cited"),
                 ("all correct articles cited", "all_correct_cited")]:
    print(f"{lbl:32s} {p20[col].sum()}/60  ->  {p21[col].sum()}/60")
f20 = p20[p20.fabricated.map(bool)]
f21 = p21[p21.fabricated.map(bool)]
print(f"{'answers naming an out-of-context art':32s} {len(f20)}/60  ->  {len(f21)}/60")
if len(f21):
    print("   (check each by hand -- Phase 20's single flag was a cross-reference quoted")
    print("    from the retrieved article's own text, i.e. a false positive)")
    print(f21[["question_id", "fabricated"]].to_string(index=False))

COVERAGE -- did the model start using more of its context?
                               Phase 20   Phase 21     delta
mean distinct articles cited       1.22       1.55     +0.33
mean answer length (chars)          463        831   +367.58
answers citing <=1 article           48         34       -14



per question: cites more 19, cites fewer 0, unchanged 41

CITATION DEFECTS
Arabic answers citing in English       8/30  ->  0/30
answers naming only the wrong law      3/60  ->  1/60   ['q03', 'q05', 'q16'] -> ['q05']

REGEX PROXY
correct article cited            55/60  ->  58/60
all correct articles cited       51/60  ->  55/60
answers naming an out-of-context art 1/60  ->  4/60
   (check each by hand -- Phase 20's single flag was a cross-reference quoted
    from the retrieved article's own text, i.e. a false positive)
question_id fabricated
        q21   [٢٦, ٣٩]
        q33       [32]
        q34       [28]
        q49       [14]


## 3. LLM judge

Same judge, same prompt, same model as Phase 20, so the two arms are directly comparable.

`max_tokens` is pinned. Left unset, OpenRouter reserves credit against the model's 65,536-token ceiling rather than actual usage and 402s a healthy balance — that killed the Phase 20 run at question 31 of 60, and because the eval set is ordered by law it left the entire Labour Law block unmeasured. Set too low (300, Phase 12b) the model's reasoning consumes the budget and content comes back empty.

In [4]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(PROJECT / ".env")
judge_client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1")

import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import JUDGE_MODEL, JUDGE_PROMPT, make_judge

judge_one = make_judge(judge_client, max_tokens=JUDGE_MAX_TOKENS)
JUDGE_MAX_TOKENS = 2000





judge_rows = []
if JUDGE_PATH.exists():
    judge_rows = pd.read_csv(JUDGE_PATH, encoding="utf-8-sig").to_dict("records")
jdone = {r["question_id"] for r in judge_rows}
print(f"{len(jdone)} already judged, {len(ev) - len(jdone)} to go")

for q in ev.itertuples(index=False):
    if q.question_id in jdone:
        continue
    ans = gen.loc[gen.question_id == q.question_id, "answer"].iloc[0]
    verdict = judge_one(q.question_text, chunks_for(q.question_id), ans)
    judge_rows.append({"question_id": q.question_id, **verdict})
    pd.DataFrame(judge_rows).to_csv(JUDGE_PATH, index=False, encoding="utf-8-sig")
    print(f"  judged {len(judge_rows)}/{len(ev)}", end="\r")

jdg = pd.DataFrame(judge_rows)
print(f"\n{len(jdg)} judged")

60 already judged, 0 to go

60 judged


## 4. Results — paired against Phase 20

Retrieval is identical by construction and is not restated here. What follows is generation only.

In [5]:
base_j = pd.read_csv(BASE_JUDGE, encoding="utf-8-sig")
cmp_ = (base_j.merge(jdg, on="question_id", suffixes=("_20", "_21"))
        .merge(ev[["question_id", "question_language", "answer_law"]], on="question_id"))
cmp_.to_csv(PROJECT / "eval" / "phase21_comparison.csv", index=False, encoding="utf-8-sig")

print("=" * 74)
print("LLM JUDGE -- Phase 20 (8 rules) vs Phase 21 (12 rules + Coverage)")
print("=" * 74)
print(f"{'':16s} {'Phase 20':>10s} {'Phase 21':>10s} {'delta':>9s}   better/worse/same")
for axis in ["faithfulness", "relevancy"]:
    a, b = cmp_[f"{axis}_score_20"], cmp_[f"{axis}_score_21"]
    up, dn = (b > a).sum(), (b < a).sum()
    print(f"{axis:16s} {a.mean():>10.3f} {b.mean():>10.3f} {b.mean() - a.mean():>+9.3f}   "
          f"{up} / {dn} / {len(a) - up - dn}")

print("\nscore distribution")
for axis in ["faithfulness", "relevancy"]:
    for arm in ["20", "21"]:
        s = cmp_[f"{axis}_score_{arm}"]
        print(f"  {axis:13s} phase{arm}  " + "  ".join(f"{v}:{(s == v).sum():2d}" for v in [5, 4, 3, 2, 1]))

print("\nby question language")
for lang, sub in cmp_.groupby("question_language"):
    print(f"  {lang}  faith {sub.faithfulness_score_20.mean():.2f} -> {sub.faithfulness_score_21.mean():.2f}"
          f"   rel {sub.relevancy_score_20.mean():.2f} -> {sub.relevancy_score_21.mean():.2f}")

print("\nby law")
for law, sub in cmp_.groupby("answer_law"):
    print(f"  {law:8s} faith {sub.faithfulness_score_20.mean():.2f} -> {sub.faithfulness_score_21.mean():.2f}"
          f"   rel {sub.relevancy_score_20.mean():.2f} -> {sub.relevancy_score_21.mean():.2f}")

LLM JUDGE -- Phase 20 (8 rules) vs Phase 21 (12 rules + Coverage)
                   Phase 20   Phase 21     delta   better/worse/same
faithfulness          4.700      4.800    +0.100   12 / 8 / 40
relevancy             4.550      4.800    +0.250   14 / 3 / 43

score distribution
  faithfulness  phase20  5:46  4:12  3: 1  2: 0  1: 1
  faithfulness  phase21  5:49  4:10  3: 1  2: 0  1: 0
  relevancy     phase20  5:41  4:12  3: 6  2: 1  1: 0
  relevancy     phase21  5:50  4: 8  3: 2  2: 0  1: 0

by question language
  ar  faith 4.70 -> 4.77   rel 4.43 -> 4.83
  en  faith 4.70 -> 4.83   rel 4.67 -> 4.77

by law
  labor96  faith 4.77 -> 4.92   rel 4.77 -> 4.88
  prof19   faith 4.75 -> 4.75   rel 4.75 -> 5.00
  ssl      faith 4.63 -> 4.70   rel 4.33 -> 4.70


In [6]:
# Regressions first -- a mean that improves while faithfulness quietly breaks on a few
# questions is exactly the Phase 12b failure, and a mean will not show it.
print("=" * 74)
print("REGRESSIONS -- questions that got WORSE")
print("=" * 74)
worse = cmp_[(cmp_.faithfulness_score_21 < cmp_.faithfulness_score_20) |
             (cmp_.relevancy_score_21 < cmp_.relevancy_score_20)]
print(f"{len(worse)} of 60\n")
for r in worse.itertuples():
    print(f"--- {r.question_id} [{r.question_language}] "
          f"faith {r.faithfulness_score_20}->{r.faithfulness_score_21}  "
          f"rel {r.relevancy_score_20}->{r.relevancy_score_21}")
    if r.faithfulness_score_21 < r.faithfulness_score_20:
        print(f"    faith: {r.faithfulness_reason_21}")
    if r.relevancy_score_21 < r.relevancy_score_20:
        print(f"    rel  : {r.relevancy_reason_21}")

print()
print("=" * 74)
print("FIXES -- questions that got BETTER")
print("=" * 74)
better = cmp_[(cmp_.faithfulness_score_21 > cmp_.faithfulness_score_20) |
              (cmp_.relevancy_score_21 > cmp_.relevancy_score_20)]
print(f"{len(better)} of 60\n")
for r in better.itertuples():
    print(f"--- {r.question_id} [{r.question_language}] "
          f"faith {r.faithfulness_score_20}->{r.faithfulness_score_21}  "
          f"rel {r.relevancy_score_20}->{r.relevancy_score_21}")

print()
print("=" * 74)
print("STILL FAILING -- below 5 in BOTH arms (the residual work list)")
print("=" * 74)
stuck = cmp_[((cmp_.faithfulness_score_20 < 5) & (cmp_.faithfulness_score_21 < 5)) |
             ((cmp_.relevancy_score_20 < 5) & (cmp_.relevancy_score_21 < 5))]
for r in stuck.itertuples():
    print(f"  {r.question_id} [{r.question_language}] faith {r.faithfulness_score_20}->"
          f"{r.faithfulness_score_21}  rel {r.relevancy_score_20}->{r.relevancy_score_21}")
    print(f"      {r.relevancy_reason_21[:150]}")

REGRESSIONS -- questions that got WORSE
10 of 60

--- q08 [en] faith 5->4  rel 5->4
    faith: All claims are drawn directly from the provided articles, though blending Social Security and Labor Law provisions without noting Article 86's applicability condition slightly overstates their combined relevance.
    rel  : The answer directly addresses coverage for delayed-onset occupational illness citing the relevant 2-year rule, but adds tangential Labor Law claim-filing details that may not apply if the person is under Social Security coverage.
--- q09 [en] faith 4->3  rel 4->4
    faith: The Article 40 quote is accurate, but the answer misapplies Article 85(b) (which concerns combining pensions with other pensions/compensation, not wages) as if it limits combining a disability pension with job income, which is not supported by the documents.
--- q10 [en] faith 5->4  rel 4->4
    faith: The answer accurately quotes Articles 45 and 70, but states unconditionally that 'you qualify' without

## Verdict — the Coverage rules worked, on relevancy only

Against the reading committed in the header: **articles-cited rose and relevancy rose, with faithfulness not falling.** That is the "worked" branch. But the two axes are not equally supported and should not be reported as one result.

### Retrieval
Unchanged by construction — both arms replay Phase 20's saved retrieval, so every difference below is the prompt and the context-formatting fix. recall@1 0.867 · recall@5 1.000 · MRR 0.922.

### Coverage — the mechanism did what it was designed to do
| | Phase 20 | Phase 21 |
|---|---|---|
| mean distinct articles cited | 1.22 | **1.55** |
| answers citing exactly one article | 48/60 | **34/60** |
| per-question movement | — | **19 cite more, 0 cite fewer** |
| mean answer length | 463 chars | 831 chars |

19–0 with nothing lost. The instruction took.

### Generation — regex proxy
correct article cited **55 → 58/60** · all correct articles cited **51 → 55/60** · **real fabrications 0 → 0**.

Flagged "fabrications" went 1 → 4, and **all four are false positives** — article numbers quoted from inside the retrieved text (q21 → 26/39, q33 → 32, q34 → 28, q49 → 14). Article 25's own text reads "entitlements specified in articles (32) and (33) of this Law", so quoting it verbatim scored as inventing Article 32. The flag count rises because longer answers quote more of the source, i.e. **it rises with faithfulness, not against it**. `grounded_article_numbers()` in `src/generation.py` now grounds cross-references, and `extract_cited_articles()` normalises Arabic-Indic digits (3 of 60 answers used them; none flipped a score here, but "المادة (٦٥)" never equals "65").

### LLM judge — paired, ties excluded
| axis | Phase 20 | Phase 21 | delta | better/worse | sign test |
|---|---|---|---|---|---|
| faithfulness | 4.700 | 4.800 | +0.100 | 12 / 8 | **p = 0.50 — noise** |
| relevancy | 4.550 | **4.800** | **+0.250** | **14 / 3** | **p = 0.013 — real** |

**Relevancy is a real gain. Faithfulness is a wash and must be reported as "held", not "improved"** — +0.100 sits exactly at the Phase 12b noise floor and 12–8 is a coin flip. Reporting the +0.100 as an improvement would be the same error as reading a 1–2 question gap off a single run.

Worst cases improved on both axes: faithfulness min 1 → 3, relevancy min 2 → 3; answers below 4 went faithfulness 2 → 1 and relevancy **7 → 2**. Perfect 5/5 on both axes: **34/60 → 43/60**.

By law, the weak arm closed most: ssl relevancy **4.33 → 4.70**, faithfulness 4.63 → 4.70. By language, Arabic relevancy **4.43 → 4.83** — the largest single movement in the run, consistent with the citation fix landing on Arabic answers.

### Citation defects
**Arabic answers citing in English: 8/30 → 0/30.** Fixed outright. Wrong law named **3/60 → 1/60** (q03 and q16 fixed; q05 survives, and it is the ambiguous one where labor96 Art 90 does arguably answer a question that names no law).

**q16 — Phase 20's worst answer — is fixed:** faithfulness 1 → 4, relevancy 2 → 5. It previously opened "نعم، يُخصم من راتبك" and then quoted that the establishment alone pays. Both the coverage rules and the Arabic context labels bear on it, and this run cannot separate them.

### The cost: Phase 12b's failure mode is attenuated, not eliminated
8 questions lost faithfulness, and **5 of the 8 are the judge describing the model relating two provisions**: q09 (applies Article 85(b), about combining *pensions with pensions*, as though it limited combining a pension with *wages*), q08 ("blending Social Security and Labor Law provisions"), q18 (imports Article 36's re-examination into an Article 35 answer), q17 (conflates the relapse window with the treatment-duration limit), q23 (implies a conditional paragraph automatically applies).

This is the predicted risk and it materialised at a measurable rate. Rule 6 held it to a level where relevancy still won clearly and faithfulness broke even — but "tell the model to cover more provisions" and "the model starts relating provisions" remain coupled. **q09 is the caution worth carrying: mid-run it looked like the flagship success, because it picked up exactly the Article 85 the Phase 20 judge said was missing. It picked it up and misapplied it.** More coverage is not automatically better coverage.

### Adopted
Both changes ship: the 12-rule prompt and the bilingual `_format_context`. Relevancy is up significantly, faithfulness is unchanged, every free metric improved or held, and the worst-case answers got materially better.

### Limits of this run
- **One run per arm.** The judge's noise floor is imported from Phase 12b (~0.1), not re-measured. It is the reason faithfulness's +0.100 is called a wash; a second identical arm would settle it.
- **Two variables changed together** — prompt and context labels. They are mostly separable by instrument (the citation counts are deterministic and the coverage counts are not language-specific), but q16's fix cannot be attributed to one or the other.
- **The judge still scores faithfulness to the retrieved context, not correctness against ground truth**, so q05 can score 5/5 on faithfulness while citing the wrong law.
- The residual work list is 10 questions below 5 in *both* arms: q01, q05, q09, q10, q19, q29, q35 are cross-article omissions the coverage rule did not reach; q42 and q49 are precision, not coverage.